# F1 Race Analytics — 01: Data Collection
**Source:** Ergast(Jolpica) Motor Racing Developer API (free, no key required)  
**Coverage:** 2018–2024 F1 seasons (7 seasons, 140+ race weekends)  
**Datasets pulled:** Race results, driver standings, constructor standings, pit stops, qualifying

---

In [ ]:
%pip install requests

In [ ]:
import requests
import pandas as pd
import time
import os

# Create data folders if they don't exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

BASE_URL = 'https://api.jolpi.ca/ergast/f1'
SEASONS = list(range(2018, 2025))  # 2018 to 2024

def fetch_ergast(endpoint, params=None):
    """Generic Ergast API fetcher with pagination handling."""
    url = f"{BASE_URL}/{endpoint}.json"
    all_data = []
    offset = 0
    limit = 100  # Jolpica works reliably at 100 per page
    
    while True:
        p = params.copy() if params else {}
        p['limit'] = limit
        p['offset'] = offset
        try:
            r = requests.get(url, params=p, timeout=15)
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"Error fetching {endpoint} at offset {offset}: {e}")
            break
        
        # Extract the total count from MRData
        mrdata = data['MRData']
        total = int(mrdata['total'])
        
        # Return full data on first page, accumulate races
        if offset == 0:
            full_data = data  # return structure intact for first call
        
        offset += limit
        if offset >= total:
            break
        time.sleep(0.3)
    
    return full_data

print('Setup complete. Starting data collection...')

Setup complete. Starting data collection...


## 1. Race Results (2018–2024)

In [10]:
all_results = []

for season in SEASONS:
    offset = 0
    limit = 100
    season_races = []
    
    while True:
        url = f"{BASE_URL}/{season}/results.json"
        r = requests.get(url, params={'limit': limit, 'offset': offset}, timeout=15)
        data = r.json()
        races = data['MRData']['RaceTable']['Races']
        total = int(data['MRData']['total'])
        season_races.extend(races)
        offset += limit
        if offset >= total:
            break
        time.sleep(0.3)
    
    for race in season_races:
        for result in race.get('Results', []):
            all_results.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'circuit': race['Circuit']['circuitName'],
                'country': race['Circuit']['Location']['country'],
                'date': race['date'],
                'driver_id': result['Driver']['driverId'],
                'driver_name': f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
                'nationality': result['Driver']['nationality'],
                'constructor': result['Constructor']['name'],
                'grid_position': int(result['grid']) if result['grid'] != '0' else None,
                'finish_position': int(result['position']) if result.get('position') else None,
                'points': float(result['points']),
                'laps_completed': int(result['laps']),
                'status': result['status'],
                'fastest_lap_rank': int(result['FastestLap']['rank']) if 'FastestLap' in result else None,
                'fastest_lap_time': result['FastestLap']['Time']['time'] if 'FastestLap' in result else None,
            })
    
    time.sleep(0.3)
    print(f"Season {season}: {len(season_races)} races fetched")

df_results = pd.DataFrame(all_results)
df_results.to_csv('../data/raw/race_results.csv', index=False)
print(f"\nRace results saved: {df_results.shape[0]} rows x {df_results.shape[1]} columns")
df_results.head(3)

Season 2018: 21 races fetched
Season 2019: 21 races fetched
Season 2020: 17 races fetched
Season 2021: 22 races fetched
Season 2022: 22 races fetched
Season 2023: 22 races fetched
Season 2024: 28 races fetched

Race results saved: 2979 rows x 17 columns


,season,round,race_name,circuit,country,date,driver_id,driver_name,nationality,constructor,grid_position,finish_position,points,laps_completed,status,fastest_lap_rank,fastest_lap_time
0,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2018-03-25,vettel,Sebastian Vettel,German,Ferrari,3.0,1,25.0,58,Finished,4.0,1:26.469
1,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2018-03-25,hamilton,Lewis Hamilton,British,Mercedes,1.0,2,18.0,58,Finished,3.0,1:26.444
2,2018,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2018-03-25,raikkonen,Kimi Räikkönen,Finnish,Ferrari,2.0,3,15.0,58,Finished,2.0,1:26.373


## 2. Qualifying Results (2018–2024)

In [11]:
all_qualifying = []

for season in SEASONS:
    offset = 0
    limit = 100
    season_races = []
    
    while True:
        url = f"{BASE_URL}/{season}/qualifying.json"
        r = requests.get(url, params={'limit': limit, 'offset': offset}, timeout=15)
        data = r.json()
        races = data['MRData']['RaceTable']['Races']
        total = int(data['MRData']['total'])
        season_races.extend(races)
        offset += limit
        if offset >= total:
            break
        time.sleep(0.3)
    
    for race in season_races:
        for q in race.get('QualifyingResults', []):
            all_qualifying.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'driver_id': q['Driver']['driverId'],
                'driver_name': f"{q['Driver']['givenName']} {q['Driver']['familyName']}",
                'constructor': q['Constructor']['name'],
                'qualifying_position': int(q['position']),
                'q1_time': q.get('Q1', None),
                'q2_time': q.get('Q2', None),
                'q3_time': q.get('Q3', None),
            })
    
    time.sleep(0.3)
    print(f"Season {season}: qualifying data fetched ({len(season_races)} rounds)")

df_qualifying = pd.DataFrame(all_qualifying)
df_qualifying.to_csv('../data/raw/qualifying.csv', index=False)
print(f"\nQualifying data saved: {df_qualifying.shape[0]} rows")
df_qualifying.head(3)

Season 2018: qualifying data fetched (21 rounds)
Season 2019: qualifying data fetched (25 rounds)
Season 2020: qualifying data fetched (17 rounds)
Season 2021: qualifying data fetched (26 rounds)
Season 2022: qualifying data fetched (22 rounds)
Season 2023: qualifying data fetched (22 rounds)
Season 2024: qualifying data fetched (28 rounds)

Qualifying data saved: 2976 rows


,season,round,race_name,driver_id,driver_name,constructor,qualifying_position,q1_time,q2_time,q3_time
0,2018,1,Australian Grand Prix,hamilton,Lewis Hamilton,Mercedes,1,1:22.824,1:22.051,1:21.164
1,2018,1,Australian Grand Prix,raikkonen,Kimi Räikkönen,Ferrari,2,1:23.096,1:22.507,1:21.828
2,2018,1,Australian Grand Prix,vettel,Sebastian Vettel,Ferrari,3,1:23.348,1:21.944,1:21.838


## 3. Pit Stop Data (2018–2024)

In [14]:
all_pitstops = []

for season in SEASONS:
    # Get all rounds with pagination
    url = f"{BASE_URL}/{season}/results.json"
    all_rounds = []
    offset = 0
    while True:
        r = requests.get(url, params={'limit': 100, 'offset': offset}, timeout=15)
        data = r.json()
        all_rounds.extend(data['MRData']['RaceTable']['Races'])
        total = int(data['MRData']['total'])
        offset += 100
        if offset >= total:
            break
    total_rounds = len(all_rounds)

    for rnd in range(1, total_rounds + 1):
        url_ps = f"{BASE_URL}/{season}/{rnd}/pitstops.json"
        r = requests.get(url_ps, params={'limit': 100}, timeout=15)
        data = r.json()
        race_info = data['MRData']['RaceTable']['Races']
        if not race_info:
            continue
        race = race_info[0]
        for ps in race.get('PitStops', []):
            duration = ps.get('duration', '')
            try:
                duration_sec = float(duration) if duration else None
            except ValueError:
                duration_sec = None
            all_pitstops.append({
                'season': season,
                'round': rnd,
                'race_name': race['raceName'],
                'driver_id': ps['driverId'],
                'stop_number': int(ps['stop']),
                'lap': int(ps['lap']),
                'duration_sec': duration_sec,
            })
        time.sleep(0.25)
    print(f"Season {season}: pit stop data fetched ({total_rounds} rounds)")

df_pitstops = pd.DataFrame(all_pitstops)
df_pitstops.to_csv('../data/raw/pit_stops.csv', index=False)
print(f"\nPit stop data saved: {df_pitstops.shape[0]} rows")
df_pitstops.head(3)

Season 2018: pit stop data fetched (21 rounds)
Season 2019: pit stop data fetched (21 rounds)
Season 2020: pit stop data fetched (17 rounds)
Season 2021: pit stop data fetched (22 rounds)
Season 2022: pit stop data fetched (22 rounds)
Season 2023: pit stop data fetched (22 rounds)
Season 2024: pit stop data fetched (28 rounds)

Pit stop data saved: 5119 rows


,season,round,race_name,driver_id,stop_number,lap,duration_sec
0,2018,1,Australian Grand Prix,brendon_hartley,1,1,22.213
1,2018,1,Australian Grand Prix,raikkonen,1,18,21.421
2,2018,1,Australian Grand Prix,hamilton,1,19,21.821


## 4. Constructor Standings (2018–2024)

In [15]:
all_constructors = []

for season in SEASONS:
    url = f"{BASE_URL}/{season}/constructorStandings.json"
    all_standings = []
    offset = 0
    
    while True:
        r = requests.get(url, params={'limit': 100, 'offset': offset}, timeout=15)
        data = r.json()
        standings_list = data['MRData']['StandingsTable']['StandingsLists']
        if standings_list:
            all_standings.extend(standings_list)
        total = int(data['MRData']['total'])
        offset += 100
        if offset >= total:
            break
        time.sleep(0.3)
    
    if not all_standings:
        print(f"Season {season}: no data found")
        continue
        
    for entry in all_standings[0]['ConstructorStandings']:
        all_constructors.append({
            'season': season,
            'position': int(entry['position']),
            'constructor': entry['Constructor']['name'],
            'nationality': entry['Constructor']['nationality'],
            'points': float(entry['points']),
            'wins': int(entry['wins']),
        })
    
    time.sleep(0.3)
    print(f"Season {season}: {len(all_standings[0]['ConstructorStandings'])} constructors fetched")

df_constructors = pd.DataFrame(all_constructors)
df_constructors.to_csv('../data/raw/constructor_standings.csv', index=False)
print(f"\nConstructor standings saved: {df_constructors.shape[0]} rows")
df_constructors

Season 2018: 10 constructors fetched
Season 2019: 10 constructors fetched
Season 2020: 10 constructors fetched
Season 2021: 10 constructors fetched
Season 2022: 10 constructors fetched
Season 2023: 10 constructors fetched
Season 2024: 10 constructors fetched

Constructor standings saved: 70 rows


,season,position,constructor,nationality,points,wins
0,2018,1,Mercedes,German,655.0,11
1,2018,2,Ferrari,Italian,571.0,6
2,2018,3,Red Bull,Austrian,419.0,4
3,2018,4,Renault,French,122.0,0
4,2018,5,Force India,Indian,111.0,0
...,...,...,...,...,...,...
65,2024,6,Alpine F1 Team,French,65.0,0
66,2024,7,Haas F1 Team,American,58.0,0
67,2024,8,RB F1 Team,Italian,46.0,0
68,2024,9,Williams,British,17.0,0


## Summary — Data Collected

In [16]:
print('=' * 50)
print('DATA COLLECTION COMPLETE')
print('=' * 50)
print(f'Race results:          {len(df_results):>6} rows')
print(f'Qualifying results:    {len(df_qualifying):>6} rows')
print(f'Pit stop records:      {len(df_pitstops):>6} rows')
print(f'Constructor standings: {len(df_constructors):>6} rows')
print(f'Seasons covered:       2018 – 2024 ({len(SEASONS)} seasons)')
print('\nAll files saved to ../data/raw/')

DATA COLLECTION COMPLETE
Race results:            2979 rows
Qualifying results:      2976 rows
Pit stop records:        5119 rows
Constructor standings:     70 rows
Seasons covered:       2018 – 2024 (7 seasons)

All files saved to ../data/raw/
